### Imports

In [1]:
from utils import *
from modules import *

import pluma.schema.outdoor as outdoor
import os
import matplotlib.pyplot as plt
import pandas as pd

Current city is copenhagen!
If you wish to change the city, please edit the value in the __init__.py file


/Users/raquel/micromamba/envs/tese/lib/python3.9/site-packages/heartpy/datautils.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


_________

## Overview of the Dataset

In [2]:
datadir = os.path.join(sourcedata, "data")

participants = [
    p for p in os.listdir(datadir)
    if p.startswith("OE") and os.path.isdir(os.path.join(datadir, p))
]

participants

['OE009',
 'OE007',
 'OE022',
 'OE012',
 'OE015',
 'OE023',
 'OE024',
 'OE002',
 'OE005',
 'OE004',
 'OE021',
 'OE019',
 'OE010',
 'OE017',
 'OE011',
 'OE018',
 'OE020']

In [3]:
def get_sessions(participant):

    participant_path = os.path.join(datadir, participant)

    return [
        s for s in os.listdir(participant_path)
        if os.path.isdir(os.path.join(participant_path, s))
    ]

In [4]:
def inspect_session(participant, session):

    session_path = os.path.join(datadir, participant, session)

    return {
        "EEG": any(f.endswith(".nedf") for f in os.listdir(session_path)),
        "Empatica": os.path.exists(os.path.join(session_path, "empatica_harp_ts.csv")),
        "Accelerometer": os.path.exists(os.path.join(session_path, "Accelerometer.csv")),
        "EyeTracking": os.path.exists(os.path.join(session_path, "PupilLabs")),
        "Microphone": os.path.exists(os.path.join(session_path, "Microphone.bin")),
        "GPS": os.path.exists(os.path.join(session_path, "UBX")),
        "LSL_streams": any(f.startswith("Streams_") for f in os.listdir(session_path)),
    }

In [5]:
rows = []

for participant in participants:

    sessions = get_sessions(participant)

    for session in sessions:

        info = inspect_session(participant, session)

        info["participant"] = participant
        info["session"] = session

        rows.append(info)

audit_df = pd.DataFrame(rows)
audit_df

,EEG,Empatica,Accelerometer,EyeTracking,Microphone,GPS,LSL_streams,participant,session
0,True,True,True,True,True,True,True,OE009,Copenhagen_Nordhavn_sub-OE203009_2024-06-25T07...
1,True,True,True,True,True,True,True,OE009,Copenhagen_Norrebro_sub-OE201009_2024-06-24T13...
2,True,True,True,True,True,True,True,OE007,Copenhagen_Hellerup_sub-OE204007_2024-06-27T09...
3,True,True,True,True,True,True,True,OE022,Copenhagen_Norrebro_sub-OE201022_2024-06-24T11...
4,True,True,True,True,True,True,True,OE022,Copenhagen_Norreport_sub-OE202022_2024-06-26T0...
5,True,True,True,True,True,True,True,OE022,Copenhagen_Nordhavn_sub-OE203022_2024-06-25T12...
6,True,True,True,True,True,True,True,OE012,Copenhagen_Norreport_sub-OE202012_2024-06-26T1...
7,False,True,True,True,True,True,True,OE015,Copenhagen_Norreport_sub-OE2002015_2024-06-28T...
8,True,True,True,True,True,True,True,OE023,Copenhagen_Norreport_sub-OE202023_2024-06-26T0...
9,True,True,True,True,True,True,True,OE023,Copenhagen_Nordhavn_sub-OE203023_2024-06-25T08...


## No ECG data in dataset?

In [12]:
def check_ecg_presence(session_path):

    session_path = os.path.join(datadir, participant, session)

    print("\nSession:", os.path.basename(session_path))

    files = os.listdir(session_path)

    # 1️⃣ Empatica CSV
    if "empatica_harp_ts.csv" in files:
        print("Empatica file found")

    # 2️⃣ HARP streams
    harp_streams = [f for f in files if f.startswith("Streams_")]
    print("HARP streams:", harp_streams)

    # 3️⃣ Look for ECG-like files
    ecg_candidates = [
        f for f in files
        if "ecg" in f.lower() or "heart" in f.lower()
    ]

    print("ECG candidate files:", ecg_candidates)

check_ecg_presence(session_path)

NameError: name 'session_path' is not defined

In [ ]:
ecg_found = False

for participant in participants:

    participant_path = os.path.join(datadir, participant)

    for session in os.listdir(participant_path):

        session_path = os.path.join(participant_path, session)

        empatica = os.path.join(session_path, "empatica_harp_ts.csv")

        if os.path.exists(empatica):

            df = pd.read_csv(empatica, nrows=5)

            if any("ecg" in c.lower() for c in df.columns):
                ecg_found = True
                print("ECG FOUND:", participant, session)

print("\nECG present anywhere:", ecg_found)


ECG present anywhere: False
